# Structure

Here we are creating the skeleton of our CSV

In [1]:
import pandas as pd

# Define the structure of the CSV
case_studies = ["Milk", "Grazing", "Fauna"]
summaries = ["BART", "BERT"]
llms = ["Claude", "GPT"]
roles = ["Yes", "No"]
examples = ["Yes", "No"]
insights = ["Yes", "No"]

data = {
    "Case study": [],
    "Summary": [],
    "LLM": [],
    "Role": [],
    "Example": [],
    "Insight": [],
    "Input": [],
    "Output": [],
    "BLEU": [],
    "METEOR": [],
    "ROUGE-1": [],
    "ROUGE-2": [],
    "ROUGE-L": [],
    "Flesch Reading Ease": [],
    "Coverage (Rater1)": [],
    "Faithfulness (Rater1)": [],
    "Coverage (Rater2)": [],
    "Faithfulness (Rater2)": [],
    
}

# Populate the data dictionary with all combinations
for case_study in case_studies:
    for summary in summaries:
        for llm in llms:
            for role in roles:
                for example in examples:
                    for insight in insights:
                        data["Case study"].append(case_study)
                        data["Summary"].append(summary)
                        data["LLM"].append(llm)
                        data["Role"].append(role)
                        data["Example"].append(example)
                        data["Insight"].append(insight)
                        data["Input"].append(None)
                        data["Output"].append(None)
                        data["BLEU"].append(None)
                        data["METEOR"].append(None)
                        data["ROUGE-1"].append(None)
                        data["ROUGE-2"].append(None)
                        data["ROUGE-L"].append(None)
                        data["Flesch Reading Ease"].append(None)
                        data["Coverage (Rater1)"].append(None)
                        data["Faithfulness (Rater1)"].append(None)
                        data["Coverage (Rater2)"].append(None)
                        data["Faithfulness (Rater2)"].append(None)
                        

# Create the DataFrame
df = pd.DataFrame(data)

# Save the DataFrame to a CSV file
csv_path = "structured_data.csv"
df.to_csv(csv_path, index=False)

print(f"CSV file created at: {csv_path}")


CSV file created at: structured_data.csv


# Filling informations

Here we are populating our CSV with the results we are having from the different CSV

In [1]:
import pandas as pd

# Define the file paths for each case study and LLM combination
file_paths = {
    ("Milk", "Claude"): 'Results/MilkCLAUDE2.csv',  # Replace with actual path
    ("Milk", "GPT"): 'Results/MilkGPT2.csv',        # Replace with actual path
    ("Grazing", "Claude"): 'Results/GrazingCLAUDE.csv',  # Replace with actual path
    ("Grazing", "GPT"): 'Results/GrazingGPT.csv',        # Replace with actual path
    ("Fauna", "Claude"): 'Results/FaunaCLAUDE.csv',      # Replace with actual path
    ("Fauna", "GPT"): 'Results/FaunaGPT.csv',            # Replace with actual path
}

# Load the structured DataFrame
structured_csv_path = 'Structure/structured_data.csv'  # Update this with your file path if different
print(f"Loading structured CSV from {structured_csv_path}")
structured_df = pd.read_csv(structured_csv_path)
print(f"Structured CSV loaded. Columns: {structured_df.columns.tolist()}")

# Mapping of columns to the metrics
metrics_mapping = {
    'Summary (BART) Reduced': {
        'BLEU (BART)': 'BLEU',
        'METEOR (BART)': 'METEOR',
        'ROUGE-1 (BART)': 'ROUGE-1',
        'ROUGE-2 (BART)': 'ROUGE-2',
        'ROUGE-L (BART)': 'ROUGE-L',
        'Flesch Reading Ease (BART)': 'Flesch Reading Ease',
    },
    'Summary (BERT) Reduced': {
        'BLEU (BERT)': 'BLEU',
        'METEOR (BERT)': 'METEOR',
        'ROUGE-1 (BERT)': 'ROUGE-1',
        'ROUGE-2 (BERT)': 'ROUGE-2',
        'ROUGE-L (BERT)': 'ROUGE-L',
        'Flesch Reading Ease (BERT)': 'Flesch Reading Ease',
    },
}

# Function to update the structured DataFrame with summaries, metrics, and context prompt from input DataFrame
def update_structured_df(input_df, structured_df, case_study, llm):
    print(f"Processing case study: {case_study}, LLM: {llm}")
    for summary_type in ['Summary (BART) Reduced', 'Summary (BERT) Reduced']:
        summary_model = summary_type.split(' ')[1].strip('()')
        print(f"Checking summary type: {summary_type} for model: {summary_model}")
        if summary_type in input_df.columns:
            print(f"Found summary column: {summary_type} in input data")
            for index, row in input_df.iterrows():
                combination_desc = row['Combination Description']
                
                # Determine if the target words are in the combination description
                role_present = "role" in combination_desc.lower()
                example_present = "example" in combination_desc.lower()
                insight_present = "insights" in combination_desc.lower()
                
                role = 'Yes' if role_present else 'No'
                example = 'Yes' if example_present else 'No'
                insight = 'Yes' if insight_present else 'No'
                
                print(f"Row {index}: role={role}, example={example}, insight={insight}")

                filter_condition = (
                    (structured_df['Case study'] == case_study) &
                    (structured_df['Summary'] == summary_model) &
                    (structured_df['LLM'] == llm) &
                    (structured_df['Role'] == role) &
                    (structured_df['Example'] == example) &
                    (structured_df['Insight'] == insight)
                )

                # Debug filter condition
                matching_rows = structured_df[filter_condition]
                print(f"Filter condition matches {len(matching_rows)} rows in structured_df")
                if len(matching_rows) == 0:
                    # Print debug information if no rows match
                    print(f"Debug info for index {index}:")
                    print(f"Case study: {case_study}")
                    print(f"Summary: {summary_model}")
                    print(f"LLM: {llm}")
                    print(f"Role: {role}")
                    print(f"Example: {example}")
                    print(f"Insight: {insight}")
                    print("Structured_df values for comparison:")
                    print(structured_df[['Case study', 'Summary', 'LLM', 'Role', 'Example', 'Insight']].drop_duplicates())

                # Update the Output column with the summary
                if summary_type in row:
                    structured_df.loc[filter_condition, 'Output'] = row[summary_type]
                    print(f"Updated Output for row {index} with summary from {summary_type}")

                # Update the Input column with the context prompt
                if 'Context Prompt' in row:
                    structured_df.loc[filter_condition, 'Input'] = row['Context Prompt']
                    print(f"Updated Input for row {index} with context prompt")

                # Update the metrics columns
                for metric, column in metrics_mapping[summary_type].items():
                    if metric in row:
                        structured_df.loc[filter_condition, column] = row[metric]
                        print(f"Updated {column} for row {index} with value from {metric}")

# Update structured DataFrame for each case study and LLM combination
for (case_study, llm), file_path in file_paths.items():
    print(f"Loading input CSV from {file_path}")
    try:
        input_df = pd.read_csv(file_path)
        print(f"Input CSV loaded. Columns: {input_df.columns.tolist()}")
        update_structured_df(input_df, structured_df, case_study, llm)
    except FileNotFoundError:
        print(f"File not found: {file_path}")
    except Exception as e:
        print(f"Error loading {file_path}: {e}")

# Save the updated DataFrame to a CSV file
updated_csv_path = "Final Sheet/updated_structured_data2.csv"
print(f"Saving updated CSV to {updated_csv_path}")
structured_df.to_csv(updated_csv_path, index=False)
print(f"Updated CSV file created at: {updated_csv_path}")


Loading structured CSV from Structure/structured_data.csv
Structured CSV loaded. Columns: ['Case study', 'Summary', 'LLM', 'Role', 'Example', 'Insight', 'Input', 'Output', 'BLEU', 'METEOR', 'ROUGE-1', 'ROUGE-2', 'ROUGE-L', 'Flesch Reading Ease', 'Coverage (Rater1)', 'Faithfulness (Rater1)', 'Coverage (Rater2)', 'Faithfulness (Rater2)']
Loading input CSV from Results/MilkCLAUDE2.csv
Input CSV loaded. Columns: ['Combination Description', 'Context Prompt', 'Context Response', 'Plot 1 Prompt', 'Plot 1 Analysis', 'Plot 2 Prompt', 'Plot 2 Analysis', 'Plot 3 Prompt', 'Plot 3 Analysis', 'Plot 4 Prompt', 'Plot 4 Analysis', 'Plot 5 Prompt', 'Plot 5 Analysis', 'Plot 6 Prompt', 'Plot 6 Analysis', 'Plot 7 Prompt', 'Plot 7 Analysis', 'Plot 8 Prompt', 'Plot 8 Analysis', 'Plot 9 Prompt', 'Plot 9 Analysis', 'Plot 10 Prompt', 'Plot 10 Analysis', 'Plot 11 Prompt', 'Plot 11 Analysis', 'Plot 12 Prompt', 'Plot 12 Analysis', 'Summary (BART) Reduced', 'Summary (BERT) Reduced', 'BLEU (BART)', 'METEOR (BART)', 